# Fase 1: Spark ML + MLflow Baseline
Este notebook establece el baseline del proyecto utilizando **Logistic Regression** sobre variables numéricas, integrando la ingesta desde **BigQuery** y el tracking con **MLflow**.

In [ ]:
import os
from pyspark.sql import SparkSession
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

## 1. Configuración de SparkSession y BigQuery
Configuramos Spark para usar el conector de BigQuery y autenticarnos con la Service Account.

In [ ]:
# Ruta al secreto montado en el contenedor
GCP_KEY_PATH = "/opt/spark/gcp-key.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_KEY_PATH

spark = SparkSession.builder \
    .appName("Fase1_SparkML_Baseline") \
    .config("spark.jars.packages", "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.35.1") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", GCP_KEY_PATH) \
    .getOrCreate()

print("Spark Session iniciada con soporte para BigQuery")

## 2. Ingesta de Datos desde BigQuery
Leemos el dataset `census_adult_income`.

In [ ]:
table = "bigquery-public-data.ml_datasets.census_adult_income"

df = spark.read.format("bigquery") \
    .option("table", table) \
    .load()

df.printSchema()
df.show(5)

## 3. Preparación de Datos y Pipeline de ML
Seleccionamos variables numéricas para el baseline.

In [ ]:
# Limpieza básica y selección de features
numeric_features = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
label_col = "income_bracket"

# Indexamos el target ( >50K -> 1, <=50K -> 0)
label_indexer = StringIndexer(inputCol=label_col, outputCol="label")

# Ensamblador de vectores
assembler = VectorAssembler(inputCols=numeric_features, outputCol="features")

# Modelo
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Pipeline
pipeline = Pipeline(stages=[label_indexer, assembler, lr])

# Split de datos
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

## 4. Entrenamiento y Tracking con MLflow
Registramos el experimento en el servidor local de MLflow.

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Spark_ML_Baseline")

with mlflow.start_run(run_name="LogisticRegression_Numerical"):
    # Log de parámetros
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("features", numeric_features)
    
    # Fit model
    model = pipeline.fit(train_df)
    
    # Predicciones
    predictions = model.transform(test_df)
    
    # Evaluación
    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    auc = evaluator.evaluate(predictions)
    
    # Log de métricas
    mlflow.log_metric("auc", auc)
    print(f"Model AUC: {auc}")
    
    # Log del modelo
    mlflow.spark.log_model(model, "model")
    
    print("Experimento registrado en MLflow")